# Nearest neighbors
There is a category of ML algorithms that memorize every training sample and the inferences of new samples are based on the distance of the sample compared to its nearest training neighbors. These approach is "nearest neighbors", works on supervised learning in classification and regression tasks, this category is a type of non-generalizing learning as we are just storing our training samples and at inference we compute the distance between a new sample and our training sample following a certain criterion.

## Algorithms

### K-Nearest neighbors (KNN)
KNN is the most popular approach, the outputs are computed choosing the $k$ nearest neighbors to the sample. The optimal choice of $k$ is highly data-dependent, small values add noise to the model but have a decision boundary evident; in contrast higher values reduces the noise but the decision boundary is vague. A good strategy to find a good value for $k$ is using cross-validation, however note that these estimator just stores the training samples and performs distance calculations between every one and the testing samples.

### Radius neighbors
Another approach establishes a radius $r$ where the samples with lower distance than $r$ are considered "close" to a sample. It can be weigh up as a more robust method to KNN and actually is because mitigates the problem with of having a lot of training samples close to the testing instance, specially when adding distance to the sample to giving more weight to closest instances.

## Implementation

### Classification
Having $n$ training samples and $c$ classes, at inference it assigns the most frequent class label from the nearest neighbors to the evaluated sample. In some cases is useful to add weights $w_i$ to every neighbor based on how close is to the sample, however the most common approach sets $w_i = 1$ giving the same treatment to every training sample.

$$
    \hat{y}(x) = \operatorname*{arg\,max}_{c \in \{1,\ldots, C \}} \sum_{i=1}^{n} w_i \mathbf{I}[y_{i} = c]
$$

where the decision function $\mathbf{I}$ is defined as follows:
$$
     \mathbf{I}[y_i = c] = \begin{cases}
 1 & \text{if } y_i = c \\
 0 & \text{otherwise}
\end{cases}
$$

### Regression
Averages the output of the neighbors to compute the output, also we can add a weight $w_i$ to give more significance to nearby points.

$$
    \hat{y}(x) = \sum_{i=1}^{n} w_i y_i
$$


In [ ]:
from pandas import read_csv
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNNClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score

data = read_csv("/kaggle/input/datasets/iabhishekofficial/mobile-price-classification/train.csv")
COL_MAPPER = {"fc": "front_camera_pixels", "four_g": "has_4g", "int_memory": "storage_size", 
              "mobile_wt": "weight", "m_dep": "depth_cm", "n_cores": "cpu_cores", "pc": "rear_camera_pixels",
              "px_width": "screen_width_pixels", "px_height": "screen_height_pixels", "sc_w": "screen_width_cm",
              "sc_h": "screen_height_cm", "three_g": "has_3g", "touch_screen": "has_touch_screen", "wifi": "has_wifi",
              "blue": "has_bluetooth", "clock_speed": "cpu_clock_speed"}

renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["battery_power","cpu_clock_speed", "depth_cm", "front_camera_pixels", "storage_size", "weight", "cpu_cores",
       "rear_camera_pixels", "screen_height_pixels","screen_width_pixels","ram","screen_height_cm","screen_width_cm","talk_time"]
BINS = ["has_4g", "has_3g", "has_bluetooth", "dual_sim", "has_wifi", "has_touch_screen"]
FEATURES = NUMS + BINS
TAG = "price_range"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

TRANSFORMER = ColumnTransformer([
    ("standarized", StandardScaler(), NUMS)
], remainder="passthrough")

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
CLASSIFICATION_MODEL = LogisticRegression(solver="saga", C=0.5, l1_ratio=0.3,random_state=123)
CLASSIFICATION_MODEL.fit(X_train, y_train)
y_pred = CLASSIFICATION_MODEL.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1Score = f1_score(y_test, y_pred, average="weighted")
print(f"Accuracy: {(accuracy*100):2f} %")
print(f"Weighted F1-Score: {(f1Score*100):2f} %")

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

data = read_csv("/kaggle/input/datasets/jsonali2003/mobile-price-prediction-dataset/Mobile Price Prediction Datatset.csv")

data = data.drop(columns=["Unnamed: 0"])
COL_MAPPER = {"Brand me": "brand", "Ratings": "user_ratings", "RAM": "ram", 
              "ROM": "storage_size", "Mobile_Size": "display_size_inches", "Primary_Cam": "main_camera_px", "Selfi_Cam": "selfie_camera_px",
              "Battery_Power": "battery", "Price": "price"}
renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["user_ratings","ram", "storage_size", "display_size_inches", "main_camera_px", "selfie_camera_px", "battery"]
CATS = ["brand"]
FEATURES = NUMS + CATS
TAG = "price"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

CAT_TRANSFORMER = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

TRANSFORMER = ColumnTransformer([
    ("num", KNNImputer(), NUMS),
    ("cat", CAT_TRANSFORMER, CATS)
])

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
REGRESSION_MODEL = LinearRegression(n_jobs=-1, fit_intercept=True)
REGRESSION_MODEL.fit(X_train, y_train)
y_pred = REGRESSION_MODEL.predict(X_test)

squared_error = mean_squared_error(y_test, y_pred)
absolute_error = mean_absolute_error(y_test, y_pred)
print(f"Mean squared error: {squared_error:2f}")
print(f"Mean absolute error: {absolute_error:2f}")